# Embeddings work
## Jose David Castillo

This notebook follows Chapter 2 of *Build a Large Language Model (From Scratch)* and reproduces the core embedding pipeline.

### Base Set-up

In [ ]:
# %pip install torch tiktoken, only run if needed
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Load the text

In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    text = f.read()

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a


### Why start with text?
In LLMs and agentive systems, everything starts with textual data. Raw text must be converted into numerical representations so that a neural network can process it. This initial step ensures that the model works with structured information rather than arbitrary symbols.

### Tokenization with tiktoken

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
tokens = tokenizer.encode(text)
print(len(tokens), tokens[:20])

5145 [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438]


### Importance of Tokenization
Tokenization divides text into manageable units (tokens). For an LLM, tokens are the interface between human language and mathematical computation. Without tokenization, the model could not learn patterns of meaning or generalize across words and phrases.

### Sliding window dataset
Transformers require fixed-length inputs. Sliding windows allow us to generate many training samples from a single text by moving a window across the token sequence.

In [8]:

class TextDataset(Dataset):
    def __init__(self, tokens, max_length, stride):
        self.inputs = []
        self.targets = []
        for i in range(0, len(tokens) - max_length, stride):
            self.inputs.append(tokens[i:i+max_length])
            self.targets.append(tokens[i+1:i+max_length+1])

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.inputs[idx], dtype=torch.long),
            torch.tensor(self.targets[idx], dtype=torch.long)
        )

### Why overlap is useful
Using a stride smaller than `max_length` creates overlapping samples. This improves data efficiency and preserves context across boundaries, which is especially important for long-range dependencies.

In [9]:
configs = [
    (32, 32),
    (32, 16),
    (64, 32),
]

for max_length, stride in configs:
    ds = TextDataset(tokens, max_length, stride)
    print(f"max_length={max_length}, stride={stride} → samples={len(ds)}")


max_length=32, stride=32 → samples=160
max_length=32, stride=16 → samples=320
max_length=64, stride=32 → samples=159


## Embeddings and meaning
Embeddings encode meaning because they are learned under prediction pressure. Tokens that appear in similar contexts receive similar gradient updates, pulling their vectors closer in space. This is a distributed representation: meaning is not in a single neuron but across dimensions. Embeddings are conceptually the first hidden layer of a neural network.

In [10]:
vocab_size = tokenizer.n_vocab
embed_dim = 64

embedding = torch.nn.Embedding(vocab_size, embed_dim)

sample = torch.tensor(tokens[:10])
embedded = embedding(sample)

embedded.shape

torch.Size([10, 64])

### Conclusion

This notebook demonstrates the essential steps required to convert raw text into meaningful numerical representations for language models. Through tokenization and fixed-length sliding windows, continuous text is transformed into input–target pairs suitable for autoregressive next-token prediction.

Embedding layers map discrete tokens into dense vector spaces that are learned via backpropagation. Embeddings encode meaning because tokens appearing in similar contexts receive similar gradient updates, causing semantic structure to emerge as geometric relationships in the embedding space.

These embeddings form the first semantic layer of a language model and provide the foundation for deeper representations, making them critical for both large language models and modern agentic systems that rely on context chunking and memory.